# CTC decoding for a toy OCR line

The fixture keeps the time-major `(T, N, V)` contract visible and decodes repeats and blank id 0 without an OCR provider.

In [ ]:
from pathlib import Path
import importlib.util
import sys
lesson_rel = Path('phases/04-computer-vision/19-ocr-document-understanding')
candidates = []
for base in (Path.cwd(), *Path.cwd().parents):
    candidates.extend((base / lesson_rel / 'code/main.py', base / 'code/main.py'))
code_path = next(p.resolve() for p in candidates if p.is_file())
spec = importlib.util.spec_from_file_location('cv04_l19_nb', code_path)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
assert spec.loader is not None
spec.loader.exec_module(module)
print(code_path)

In [ ]:
if not module.TORCH_AVAILABLE:
    import numpy as np
    blank = 0
    ids = [blank, 11, 11, blank, 12]
    log_probs = np.full((len(ids), 1, len(module.VOCAB)), -8.0)
    for step, index in enumerate(ids): log_probs[step, 0, index] = 0.0
    log_probs -= np.log(np.exp(log_probs).sum(axis=-1, keepdims=True))
    decoded = module.numpy_ctc_greedy_decode(log_probs)
    loss = module.numpy_ctc_loss(log_probs, np.asarray([11, 12]), np.asarray([len(ids)]), np.asarray([2]))
    assert decoded == [[11, 12]] and np.isfinite(loss)
    print({'Build-It': 'NumPy CTC', 'decoded_ids': decoded, 'loss': loss, 'Use-It': 'PyTorch skipped cleanly'})
else:
    import torch
    blank = 0
    log_probs = torch.full((5, 1, len(module.VOCAB)), -8.0)
    log_probs[:, 0, blank] = 0.0
    log_probs[1, 0, 11] = 0.0
    log_probs[3, 0, 12] = 0.0
    decoded = module.greedy_ctc_decode(log_probs)
    assert decoded == [[11, 12]]
    print({'decoded_ids': decoded, 'text': module.decode_to_str(decoded[0])})

The decoder example is deliberately small: real document understanding still needs layout and field-level validation after recognition.